### GPT-4 Tokenizer (tiktoken) Demonstration

This notebook demonstrates how to inspect, encode, decode, and analyze text using OpenAI's tiktoken library (GPT-4 cl100k_base encoding).

***Prerequisites***
Install the required packages before running:
```bash
pip install tiktoken matplotlib numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tiktoken

# Load the base BPE tokenizer for GPT-4 (100k vocabulary)
tokenizer = tiktoken.get_encoding("cl100k_base")

# Check the total vocabulary size
vocab_size = tokenizer.n_vocab
print(f"Loaded Tokenizer: {tokenizer.name}")
print(f"Total Vocabulary Size: {vocab_size} tokens")

Invertibility & Decoding : Decoding a single token ID into its corresponding string representation.

In [ ]:
sample_token_id = 1234
decoded_string = tokenizer.decode([sample_token_id])

print(f"Token ID {sample_token_id} decodes to: '{decoded_string}'")
print("(Notice the leading space included directly in the subword token!)")

Sensitivity to Spaces and Casing

In [ ]:
text_no_space = "John"
text_with_space = " John"

ids_no_space = tokenizer.encode(text_no_space)
ids_with_space = tokenizer.encode(text_with_space)

print(f"'{text_no_space}' -> Token IDs: {ids_no_space}")
print(f"'{text_with_space}' -> Token IDs: {ids_with_space}")

words_to_test = [" lanka", "Lanka", "LANKA"]
print("\nCapitalization and Subword Fragmentation:")
for word in words_to_test:
    encoded_ids = tokenizer.encode(word)
    decoded_tokens = [tokenizer.decode([tid]) for tid in encoded_ids]
    print(f"  '{word}' -> IDs: {encoded_ids} | Subwords: {decoded_tokens}")

Tokenizing Raw Uncleaned Text

In [ ]:
sample_raw_text = """
"My MFA seems to have gone on an unexpected vacation." 😅

Could you help bring it back when you get a chance?
"""

raw_text_tokens = tokenizer.encode(sample_raw_text)
print(f"Raw Text Sample:\n{sample_raw_text.strip()}\n")
print(f"Encoded Token Count: {len(raw_text_tokens)}")

print("\nToken-by-Token Breakdown (First 20 tokens):")
for tid in raw_text_tokens[:20]:
    token_str = repr(tokenizer.decode([tid]))
    print(f"  ID {tid:>6} -> {token_str}")

Analyzing Vocabulary Token Lengths

In [ ]:
print("Calculating byte lengths across vocabulary... (this may take a few seconds)")

token_lengths = []
for i in range(vocab_size):
    try:
        token_bytes = tokenizer.decode_bytes([i])
        token_lengths.append(len(token_bytes))
    except Exception:
        token_lengths.append(np.nan)

token_lengths_arr = np.array(token_lengths)
valid_lengths = token_lengths_arr[~np.isnan(token_lengths_arr)]

print(f"Valid Tokens Analyzed: {len(valid_lengths)}")
print(f"Min Token Length: {int(np.min(valid_lengths))} bytes")
print(f"Max Token Length: {int(np.max(valid_lengths))} bytes")
print(f"Mean Token Length: {np.mean(valid_lengths):.2f} bytes")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(valid_lengths, bins=50, color="skyblue", edgecolor="black", alpha=0.7)
axes[0].set_title("Token Length Distribution (Linear Scale)")
axes[0].set_xlabel("Token Length (Bytes)")
axes[0].set_ylabel("Frequency")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].hist(valid_lengths, bins=50, color="salmon", edgecolor="black", alpha=0.7, log=True)
axes[1].set_title("Token Length Distribution (Log Scale)")
axes[1].set_xlabel("Token Length (Bytes)")
axes[1].set_ylabel("Frequency (Log Scale)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()